# 01 · Data preparation

**Goal:** turn the raw M5 files into five clean tables in a local DuckDB database, ready for the SQL analysis.

| Step | What happens |
|---|---|
| 1 | Read two years of daily sales for the Household and Hobbies ranges |
| 2 | Reshape the sales from wide (one column per day) to long (one row per item, store and day) |
| 3 | Build the item, store, calendar and price tables |
| 4 | Check the data before any analysis |
| 5 | Load everything into `data/inventory.duckdb` |

**Data:** [M5 Forecasting – Accuracy](https://www.kaggle.com/c/m5-forecasting-accuracy/data) (Walmart, via Kaggle). Put `sales_train_evaluation.csv`, `calendar.csv` and `sell_prices.csv` in `data/raw/`.

In [1]:
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
DB_PATH = ROOT / "data" / "inventory.duckdb"

CATEGORIES = ["HOUSEHOLD", "HOBBIES"]  # the non-food ranges, closest to a home-improvement retailer
DAYS_TO_KEEP = 730                      # the last two years of the data

## 1. Read the sales file
The file has one row per item and store, and one column per day (`d_1` to `d_1941`). Reading only the last 730 day columns, stored as small integers, keeps memory low.

In [2]:
sales_path = RAW / "sales_train_evaluation.csv"
header = pd.read_csv(sales_path, nrows=0).columns
day_cols = [c for c in header if c.startswith("d_")]
keep_days = day_cols[-DAYS_TO_KEEP:]
id_cols = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]

wide = pd.read_csv(sales_path, usecols=id_cols + keep_days, dtype={d: "int16" for d in keep_days})
wide = wide[wide["cat_id"].isin(CATEGORIES)].reset_index(drop=True)

print(f"Days in file: {len(day_cols):,}; kept the last {len(keep_days)} ({keep_days[0]} to {keep_days[-1]})")
print(f"Household and Hobbies series: {len(wide):,} item-store pairs")
wide.groupby("cat_id")["item_id"].nunique().rename("items")

Days in file: 1,941; kept the last 730 (d_1212 to d_1941)
Household and Hobbies series: 16,120 item-store pairs


cat_id
HOBBIES       565
HOUSEHOLD    1047
Name: items, dtype: int64

## 2. Reshape wide to long
`melt` turns the 730 day columns into rows. The calendar maps each `d_` column to a real date. (Kaggle's calendar has a `d` column; some copies of M5 do not, so it is rebuilt from the row order if missing.)

In [3]:
calendar = pd.read_csv(RAW / "calendar.csv", parse_dates=["date"])
if "d" not in calendar.columns:
    calendar["d"] = "d_" + (calendar.index + 1).astype(str)
d_to_date = calendar.set_index("d")["date"]

sales_daily = wide.melt(
    id_vars=["item_id", "store_id"],
    value_vars=keep_days,
    var_name="d",
    value_name="units",
)
sales_daily["date"] = sales_daily["d"].map(d_to_date)
sales_daily = sales_daily[["item_id", "store_id", "date", "units"]]

print(f"{len(sales_daily):,} rows, {sales_daily['date'].min():%d %b %Y} to {sales_daily['date'].max():%d %b %Y}")
sales_daily.head()

11,767,600 rows, 24 May 2014 to 22 May 2016


,item_id,store_id,date,units
0,HOBBIES_1_001,CA_1,2014-05-24,0
1,HOBBIES_1_002,CA_1,2014-05-24,2
2,HOBBIES_1_003,CA_1,2014-05-24,0
3,HOBBIES_1_004,CA_1,2014-05-24,2
4,HOBBIES_1_005,CA_1,2014-05-24,1


## 3. Build the item, store, calendar and price tables

In [4]:
items = wide[["item_id", "dept_id", "cat_id"]].drop_duplicates().sort_values("item_id").reset_index(drop=True)
stores = wide[["store_id", "state_id"]].drop_duplicates().sort_values("store_id").reset_index(drop=True)

window = calendar["date"].between(sales_daily["date"].min(), sales_daily["date"].max())
calendar = calendar.loc[window, [
    "date", "wm_yr_wk", "weekday", "wday", "month", "year",
    "event_name_1", "event_type_1", "event_name_2", "event_type_2",
    "snap_CA", "snap_TX", "snap_WI",
]].reset_index(drop=True)

prices = pd.read_csv(
    RAW / "sell_prices.csv",
    dtype={"store_id": "string", "item_id": "string", "wm_yr_wk": "int32", "sell_price": "float64"},
)
prices = prices[prices["item_id"].isin(items["item_id"]) & prices["wm_yr_wk"].isin(calendar["wm_yr_wk"])]
prices = prices.reset_index(drop=True)

print(f"items: {len(items):,} | stores: {len(stores)} | calendar days: {len(calendar)} | weekly prices: {len(prices):,}")

items: 1,612 | stores: 10 | calendar days: 730 | weekly prices: 1,668,817


## 4. Check the data before analysing it
Every check should pass. Store closures and range changes are checked in SQL (`sql/02_quality_checks.sql`).

In [5]:
expected_rows = len(wide) * DAYS_TO_KEEP
checks = {
    "one row per item, store and day": len(sales_daily) == expected_rows,
    "no duplicate item-store-day keys": not sales_daily.duplicated(["item_id", "store_id", "date"]).any(),
    "no missing values in sales": not sales_daily.isna().any().any(),
    "no negative unit sales": (sales_daily["units"] >= 0).all(),
    "every day has a calendar row": sales_daily["date"].nunique() == len(calendar) == DAYS_TO_KEEP,
    "one price per item, store and week": not prices.duplicated(["item_id", "store_id", "wm_yr_wk"]).any(),
    "all prices positive": (prices["sell_price"] > 0).all(),
}
for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}  {name}")
assert all(checks.values()), "A data check failed: fix it before running the SQL."

PASS  one row per item, store and day
PASS  no duplicate item-store-day keys
PASS  no missing values in sales
PASS  no negative unit sales
PASS  every day has a calendar row
PASS  one price per item, store and week
PASS  all prices positive


A first look: how many listed item-store days have no sales at all? Zero-heavy, intermittent demand is normal in retail, and it is why stockouts have to be inferred carefully.

In [6]:
zero_share = (sales_daily["units"] == 0).mean()
print(f"Item-store days with zero sales: {zero_share:.1%}")
sales_daily["units"].describe(percentiles=[0.5, 0.9, 0.99]).round(2)

Item-store days with zero sales: 67.7%


count    11767600.00
mean            0.79
std             2.10
min             0.00
50%             0.00
90%             2.00
99%             9.00
max           567.00
Name: units, dtype: float64

## 5. Load into DuckDB
DuckDB is a free, fast SQL engine that runs inside Python with no server. The SQL scripts in `sql/` run against this file.

In [7]:
DB_PATH.unlink(missing_ok=True)
con = duckdb.connect(str(DB_PATH))
for name, df in {
    "sales_daily": sales_daily,
    "items": items,
    "stores": stores,
    "calendar": calendar,
    "prices": prices,
}.items():
    con.register("df_view", df)
    con.execute(f"CREATE TABLE {name} AS SELECT * FROM df_view")
    con.unregister("df_view")

con.execute("ALTER TABLE sales_daily ALTER date TYPE DATE")
con.execute("ALTER TABLE calendar ALTER date TYPE DATE")
con.sql("""
    SELECT table_name, estimated_size AS rows
    FROM duckdb_tables()
    ORDER BY table_name
""").df()

,table_name,rows
0,calendar,730
1,items,1612
2,prices,1668817
3,sales_daily,11767600
4,stores,10


In [8]:
con.close()
print(f"Saved {DB_PATH.relative_to(ROOT)} ({DB_PATH.stat().st_size / 1e6:,.0f} MB). Next: notebooks/02_sql_analysis.ipynb")

Saved data/inventory.duckdb (47 MB). Next: notebooks/02_sql_analysis.ipynb
